# TontumaBot V3 - Notebook de demonstration completeCe notebook presente et teste chaque modele et chaque etape du pipeline TontumaBot V3.Il est destine au coach et a l'equipe de developpement.

## 1. Objectif du notebook- Comprendre et valider chaque modele (NLLB, STT, TTS, embeddings, reranker, LLM).- Telecharger le dataset de documents depuis Hugging Face.- Executer le pipeline RAG etape par etape.- Tester les endpoints REST.- Preparer le deploiement.

## 2. Preparation de l'environnementOn ajoute le dossier src/ au PYTHONPATH.

In [ ]:
import osimport sysBASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))sys.path.insert(0, os.path.join(BASE_DIR, 'src'))print('BASE_DIR :', BASE_DIR)

## 3. Chargement de la configurationLa configuration est centralisee dans src/config.py et lue depuis le fichier .env.

In [ ]:
from config import settingsprint('LLM_PROVIDER     :', settings.LLM_PROVIDER)print('LOCAL_LLM_MODEL  :', settings.LOCAL_LLM_MODEL)print('NLLB WO->FR      :', settings.NLLB_WO_FR_MODEL)print('NLLB FR->WO      :', settings.NLLB_FR_WO_MODEL)print('STT_MODEL_PATH   :', settings.STT_MODEL_PATH)print('EMBED_MODEL      :', settings.EMBED_MODEL)print('RERANKER_MODEL   :', settings.RERANKER_MODEL)print('TTS_ENGINE       :', settings.TTS_ENGINE)print('GROQ_API_KEY set?:', bool(settings.GROQ_API_KEY))

## 4. NLLB : Traduction Wolof -> FrancaisPourquoi : la question wolof doit etre traduite en francais avant le RAG.Modele : bilalfaye/nllb-200-distilled-600M-wo-fr-en (fine-tune wolof-francais).

In [ ]:
from translation.nllb import wolof_to_frenchwo_text = 'Naka nga ame kayitu juddu gi?'fr_text, latency = wolof_to_french(wo_text)print('Wolof  :', wo_text)print('Francais:', fr_text)print('Latence :', f'{latency:.2f} s')

## 5. NLLB : Traduction Francais -> WolofPourquoi : la reponse francaise est re-traduite en wolof si la question etait en wolof.Modele : Lahad/nllb200-francais-wolof.

In [ ]:
from translation.nllb import french_to_woloffr_response = "La sage-femme remplit le certificat d'accouchement."wo_response, latency = french_to_wolof(fr_response)print('Francais:', fr_response)print('Wolof   :', wo_response)print('Latence :', f'{latency:.2f} s')

## 6. STT : Transcription audio wolofModele : M9and2M/whisper-small-wolof (fine-tune Whisper sur wolof).Execution locale pour confidentialite.

In [ ]:
from input.stt import transcribeaudio_path = os.path.join(BASE_DIR, 'uploads', 'micro.webm')if os.path.exists(audio_path):    transcription = transcribe(audio_path, language='wo')    print('Transcription :', transcription)else:    print('Fichier non trouve :', audio_path)

## 7. Embeddings et RerankerEmbeddings : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.Reranker : cross-encoder/ms-marco-MiniLM-L-6-v2.

In [ ]:
from sentence_transformers import SentenceTransformerfrom retrieval.reranker import Rerankerprint('Chargement embedder...')embedder = SentenceTransformer(settings.EMBED_MODEL)print('OK')print('Chargement reranker...')reranker = Reranker(settings.RERANKER_MODEL)print('OK')question = 'Comment obtenir un extrait de naissance ?'passages = [    'La sage-femme remplit la declaration de naissance a la maternite.',    'Le guichet CMU est au rez-de-chaussee du batiment D.']scores = reranker.rerank(question, passages)print('Scores :', scores)

## 8. Ingestion depuis Hugging FaceLe dataset HF centralise les documents administratifs.Le serveur les telecharge automatiquement au demarrage si HF_DOCUMENTS_DATASET est defini.

In [ ]:
import osHF_DATASET = os.getenv('HF_DOCUMENTS_DATASET', '')print('HF_DOCUMENTS_DATASET :', HF_DATASET)if HF_DATASET:    from ingestion import ingest_huggingface_dataset    n = ingest_huggingface_dataset(HF_DATASET)    print('Chunks indexes :', n)else:    print('Aucun dataset HF configure. Utilisation des fichiers locaux.')

## 9. Pipeline RAG completOn appelle pipeline.answer() qui execute toutes les etapes :detection langue -> traduction -> RAG -> LLM -> traduction retour.

In [ ]:
from pipeline import answerquestion = 'Quels documents pour un extrait de naissance ?'result = answer(question, tts=False)print('--- REPONSE ---')print(result['response_fr'][:800])print('\n--- TRACE ---')print('Langue :', result['trace']['input_lang'])print('Intention :', result['trace']['intent'])print('Chunks :', result['trace']['retrieval'].get('n_filtered', 'N/A'))print('Context quality :', result['trace'].get('context_quality', {}))

## 10. Test TTS OolelAttention : Oolel est lent sur CPU. Utilisez edge-tts pour des tests rapides en francais.

In [ ]:
from tts_Ooleil.tts import synthesize, sourcetext_wo = 'Dama bëgg am kayitu juddu gi.'out_path = os.path.join(BASE_DIR, 'notebooks', 'demo_tts.wav')print('Synthese en cours...')audio = synthesize(text_wo, out_path, engine='oolel')print('Fichier :', audio)print('Moteur  :', source('oolel'))

## 11. Test des endpoints RESTLe serveur FastAPI doit etre lance sur http://localhost:8008.

In [ ]:
import requestsBASE_URL = 'http://localhost:8008'r = requests.get(f'{BASE_URL}/health')print('Health :', r.status_code, r.json())r = requests.get(f'{BASE_URL}/models')print('Models :', r.status_code)r = requests.post(f'{BASE_URL}/ask', json={'question': 'Ou est la pharmacie ?'})print('Ask :', r.status_code)print('Reponse :', r.json()['response_fr'][:300])

## 12. Resume des modeles et endpoints| Modele | Role ||--------|------|| qwen/qwen3.8-27b | Generation reponse (Groq) || nllb-wo-fr | Traduction wolof -> francais || nllb-fr-wo | Traduction francais -> wolof || whisper-small-wolof | Transcription audio || paraphrase-multilingual-MiniLM | Embeddings || ms-marco-MiniLM | Reranker || Oolel-Voices | Synthese vocale wolof || Endpoint | Methode ||----------|---------|| /health | GET || /models | GET || /ask | POST || /ask/audio | POST || /admin/documents | POST/GET |

In [ ]:
print('Fin de la demonstration. Consultez DEPLOY.md pour le deploiement.')